# MedTrack_DV — 05. KPI Engineering

Computes the 6 mandatory KPIs from the project doc, each with a documented formula and source table -
required before any Tableau dashboard work begins.

| KPI | Formula | Source |
|---|---|---|
| Total Admissions | `COUNTD(admission_id)` | hospital_overview_dataset |
| Occupancy Rate | `sum(occupied_beds_count) / sum(total_beds) x 100` | department_analytics_dataset |
| Average LOS | `mean(discharge_date - admission_date)` | hospital_overview_dataset |
| Readmission Rate | 30-day same-patient proxy (documented, not ground truth) | hospital_overview_dataset |
| Bed Utilization Rate | `sum(units_in_use) / sum(total_units_available) x 100` | resource_utilization_dataset |
| Department Efficiency Score | weighted composite, see below | department_analytics_dataset |

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

hospital_overview_dataset    = pd.read_csv(PROCESSED_DIR / "hospital_overview_dataset.csv")
department_analytics_dataset = pd.read_csv(PROCESSED_DIR / "department_analytics_dataset.csv")
resource_utilization_dataset = pd.read_csv(PROCESSED_DIR / "resource_utilization_dataset.csv")

hospital_overview_dataset['admission_date'] = pd.to_datetime(hospital_overview_dataset['admission_date'])
hospital_overview_dataset['discharge_date'] = pd.to_datetime(hospital_overview_dataset['discharge_date'])

kpi_results = []

## KPI 1 — Total Admissions

In [2]:
# Formula: COUNTD(admission_id) - counts distinct admissions, never Patient
# Flow rows (which would double-count, since that table has 2 rows/admission).
total_admissions = hospital_overview_dataset['admission_id'].nunique()
kpi_results.append({'kpi': 'Total Admissions', 'value': total_admissions, 'unit': 'count',
                     'formula': 'COUNTD(admission_id)', 'source': 'hospital_overview_dataset'})
print(f"Total Admissions: {total_admissions}")

Total Admissions: 45000


## KPI 2 — Occupancy Rate

In [3]:
# Formula: sum(occupied_beds_count) / sum(total_beds) x 100 - a
# capacity-weighted average, not a simple mean of daily percentages
# (which would treat a 6-bed and a 90-bed department as equally important).
occupancy_rate = (department_analytics_dataset['occupied_beds_count'].sum()
                   / department_analytics_dataset['total_beds'].sum() * 100)
kpi_results.append({'kpi': 'Occupancy Rate', 'value': round(occupancy_rate, 2), 'unit': '%',
                     'formula': 'sum(occupied_beds_count) / sum(total_beds) x 100',
                     'source': 'department_analytics_dataset'})
print(f"Occupancy Rate: {occupancy_rate:.2f}%")

Occupancy Rate: 30.28%


## KPI 3 — Average Length of Stay

In [4]:
# Formula: mean(discharge_date - admission_date), admission-level as the doc requires.
los_days = (hospital_overview_dataset['discharge_date'] - hospital_overview_dataset['admission_date']).dt.days
avg_los = los_days.mean()
kpi_results.append({'kpi': 'Average Length of Stay', 'value': round(avg_los, 2), 'unit': 'days',
                     'formula': 'mean(discharge_date - admission_date)', 'source': 'hospital_overview_dataset'})
print(f"Average Length of Stay: {avg_los:.2f} days")

Average Length of Stay: 5.16 days


## KPI 4 — Readmission Rate

Definition used: `readmission_flag = 1` if the SAME patient has a prior discharge within 30 days of
this admission's start date. This is a documented **proxy** built from HMIS alone - not the Readmission
dataset's real flag, since no patient-level link exists between HMIS and that dataset (it can only
contribute a disease-level benchmark, already merged into Hospital Overview in `03_data_normalization`).
Eligible population = every admission (first-ever admissions simply score 0).

In [5]:
readmission_rate = hospital_overview_dataset['readmission_flag'].mean() * 100
kpi_results.append({'kpi': 'Readmission Rate', 'value': round(readmission_rate, 2), 'unit': '%',
                     'formula': 'count(readmission_flag=1) / count(all admissions) x 100 [30-day same-patient proxy]',
                     'source': 'hospital_overview_dataset'})
print(f"Readmission Rate: {readmission_rate:.2f}% (30-day proxy, documented - not ground truth)")

Readmission Rate: 2.19% (30-day proxy, documented - not ground truth)


## KPI 5 — Bed Utilization Rate

In [6]:
# Source is restricted to Bed rows - the only resource_type this project can
# actually build (see the known Equipment/Staff gaps documented in 03).
bed_rows = resource_utilization_dataset[resource_utilization_dataset['resource_type'] == 'Bed']
bed_utilization_rate = (bed_rows['units_in_use'].sum() / bed_rows['total_units_available'].sum() * 100)
kpi_results.append({'kpi': 'Bed Utilization Rate', 'value': round(bed_utilization_rate, 2), 'unit': '%',
                     'formula': 'sum(units_in_use) / sum(total_units_available) x 100 [Bed resource_type only]',
                     'source': 'resource_utilization_dataset'})
print(f"Bed Utilization Rate: {bed_utilization_rate:.2f}%")
print("NOTE: this is numerically identical to Occupancy Rate (KPI 2), because Resource")
print("Utilization's only resource_type (Bed) was itself derived from the same occupancy")
print("numbers - expected given the current data sources, not a calculation error.")

Bed Utilization Rate: 30.28%
NOTE: this is numerically identical to Occupancy Rate (KPI 2), because Resource
Utilization's only resource_type (Bed) was itself derived from the same occupancy
numbers - expected given the current data sources, not a calculation error.


## KPI 6 — Department Efficiency Score

A documented composite 0-100 score per department, built ONLY from components this project can
reliably calculate. Several conventional inputs (mortality, equipment downtime) are unavailable from
any of the 3 sources and are excluded rather than guessed.

**Components** (each independently normalized to a 0-100 "higher is better" scale), combined as a
weighted average that **automatically re-weights** when a component is missing for a department
(never fills a missing component with a guessed value):

- **Occupancy fit (30%)**: `100 - |occupancy_pct - 80|`, floored at 0. Efficiency peaks near 80%
  occupancy - too low wastes capacity, too high risks overcrowding.
- **LOS efficiency (30%)**: departments with a shorter average LOS *relative to the other
  departments* score higher. A relative ranking across departments, not an absolute clinical judgment.
- **Readmission (25%)**: `100 - readmission_rate_pct`. Lower readmission = higher score.
- **Satisfaction (15%)**: `avg_satisfaction_score` directly, only for the 4 departments the Beds
  Management bridge actually matched (Emergency, Surgery, ICU, Internal Medicine). Pediatrics/
  Orthopedics use the other 3 components only, re-weighted to 100%.

In [7]:
dept_summary = department_analytics_dataset.groupby(['department_id', 'department_name']).agg(
    avg_occupancy_pct=('bed_occupancy_rate_pct', 'mean'),
    avg_los_days=('avg_length_of_stay_days', 'mean'),
    avg_readmission_rate_pct=('readmission_rate_pct', 'mean'),
    avg_satisfaction_score=('avg_satisfaction_score', 'mean'),
).reset_index()

dept_summary['occupancy_score'] = (100 - (dept_summary['avg_occupancy_pct'] - 80).abs()).clip(lower=0)

los_min, los_max = dept_summary['avg_los_days'].min(), dept_summary['avg_los_days'].max()
dept_summary['los_score'] = 100 - ((dept_summary['avg_los_days'] - los_min) / (los_max - los_min) * 100)

dept_summary['readmission_score'] = 100 - dept_summary['avg_readmission_rate_pct']
dept_summary['satisfaction_score'] = dept_summary['avg_satisfaction_score']

component_weights = {'occupancy_score': 0.30, 'los_score': 0.30, 'readmission_score': 0.25, 'satisfaction_score': 0.15}

def weighted_efficiency(row):
    available = {c: w for c, w in component_weights.items() if pd.notna(row[c])}
    if not available:
        return pd.NA
    weight_sum = sum(available.values())
    return sum(row[c] * (w / weight_sum) for c, w in available.items())

dept_summary['department_efficiency_score'] = dept_summary.apply(weighted_efficiency, axis=1).round(2)

print(dept_summary[['department_name', 'occupancy_score', 'los_score', 'readmission_score',
                     'satisfaction_score', 'department_efficiency_score']].round(2).to_string(index=False))

kpi_results.append({'kpi': 'Department Efficiency Score', 'value': 'see department_efficiency_scores.csv', 'unit': '0-100 scale',
                     'formula': '30% occupancy-fit + 30% LOS-rank + 25% readmission + 15% satisfaction (re-weighted when a component is unavailable)',
                     'source': 'department_analytics_dataset'})

  department_name  occupancy_score  los_score  readmission_score  satisfaction_score  department_efficiency_score
        Emergency            50.23      98.83              97.79               77.98                        80.86
Internal Medicine            50.38      99.78              97.61               80.84                        81.58
          Surgery            48.97      99.02              97.80               79.08                        80.71
       Pediatrics            51.13      98.95              97.88                 NaN                        81.76
      Orthopedics            50.52     100.00              98.00                 NaN                        81.95
              ICU            50.97       0.00              97.76               81.60                        51.97


## Save KPI Outputs

In [8]:
kpi_summary_df = pd.DataFrame(kpi_results)
kpi_summary_df.to_csv(PROCESSED_DIR / "kpi_summary.csv", index=False)
dept_summary.to_csv(PROCESSED_DIR / "department_efficiency_scores.csv", index=False)

print(f"Saved -> {PROCESSED_DIR / 'kpi_summary.csv'}")
print(f"Saved -> {PROCESSED_DIR / 'department_efficiency_scores.csv'}")
display(kpi_summary_df)

Saved -> C:\Users\sr189\OneDrive\Desktop\Hospital Management Data Analysis\MedTrack_DV\data\processed\kpi_summary.csv
Saved -> C:\Users\sr189\OneDrive\Desktop\Hospital Management Data Analysis\MedTrack_DV\data\processed\department_efficiency_scores.csv


,kpi,value,unit,formula,source
0,Total Admissions,45000,count,COUNTD(admission_id),hospital_overview_dataset
1,Occupancy Rate,30.28,%,sum(occupied_beds_count) / sum(total_beds) x 100,department_analytics_dataset
2,Average Length of Stay,5.16,days,mean(discharge_date - admission_date),hospital_overview_dataset
3,Readmission Rate,2.19,%,count(readmission_flag=1) / count(all admissio...,hospital_overview_dataset
4,Bed Utilization Rate,30.28,%,sum(units_in_use) / sum(total_units_available)...,resource_utilization_dataset
5,Department Efficiency Score,see department_efficiency_scores.csv,0-100 scale,30% occupancy-fit + 30% LOS-rank + 25% readmis...,department_analytics_dataset
